# 🎬 AnimeEncoderBot
**GPU-accelerated video encoding (AV1/HEVC) + AI anime upscaling**

⚠️ Make sure GPU T4 is enabled: **Settings → Accelerator → GPU T4 x2**

Just hit **Run All** — everything is automated.

In [ ]:
# ═══ Step 1: Check GPU ═══
!nvidia-smi
print('\n' + '='*60)
!ffmpeg -version 2>/dev/null | head -1 || echo 'FFmpeg not found'
print('='*60)

In [ ]:
# ═══ Step 2: Load Secrets ═══
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ['BOT_TOKEN']       = secrets.get_secret('BOT_TOKEN')
os.environ['API_ID']          = secrets.get_secret('API_ID')
os.environ['API_HASH']        = secrets.get_secret('API_HASH')
os.environ['ADMIN_IDS']       = secrets.get_secret('ADMINS')
os.environ['LOG_CHANNEL']     = secrets.get_secret('LOG_CHANNEL')
os.environ['GDRIVE_FOLDER_ID']= secrets.get_secret('GDRIVE_FOLDER_ID')

# GDrive SA JSON — write to file
sa_json = secrets.get_secret('GDRIVE_SA_JSON')
with open('/kaggle/working/sa.json', 'w') as f:
    f.write(sa_json)
os.environ['GDRIVE_SA_JSON'] = '/kaggle/working/sa.json'

# MongoDB — use local MongoDB for Kaggle
os.environ['MONGO_URI']       = 'mongodb://localhost:27017/anime_encoder_bot'
os.environ['GPU_ENABLED']     = 'true'
os.environ['CONCURRENT_TASKS']= '2'
os.environ['DOWNLOAD_DIR']    = '/kaggle/working/downloads'

!mkdir -p /kaggle/working/downloads
print('✅ Secrets loaded')

In [ ]:
# ═══ Step 3: Install MongoDB & System Dependencies ═══
import subprocess
import os
import glob

cmds = [
    'apt-get update -qq',
    'apt-get install -y -qq gnupg curl libvulkan1 vulkan-tools libgomp1',
]
for cmd in cmds:
    subprocess.run(cmd, shell=True, capture_output=True)
print('📦 Base packages and Vulkan dependencies installed')

# Configure NVIDIA Vulkan ICD for Real-ESRGAN GPU acceleration
# Kaggle containers have NVIDIA drivers but often lack the Vulkan ICD config.
# We need to: 1) find/install the Vulkan producer lib, 2) register it.
!mkdir -p /etc/vulkan/icd.d
import json, re

# Get driver version for targeted package install
drv_info = subprocess.run('cat /proc/driver/nvidia/version', shell=True, capture_output=True, text=True).stdout
drv_match = re.search(r'(\d+)\.\d+\.\d+', drv_info)
drv_major = drv_match.group(1) if drv_match else None
print(f'🔍 NVIDIA driver major version: {drv_major}')

# Broad search for any NVIDIA Vulkan-capable library on the system
find_result = subprocess.run(
    'find / -name "libnvidia-vulkan-producer.so*" -o -name "libGLX_nvidia.so*" '
    '-o -name "libEGL_nvidia.so*" 2>/dev/null | head -20',
    shell=True, capture_output=True, text=True
).stdout.strip()
print(f'🔍 Found NVIDIA libs:\n{find_result or "  (none)"}')

nvidia_vulkan_lib = None
# Prefer vulkan-producer, then GLX, then EGL
for prefix in ['libnvidia-vulkan-producer', 'libGLX_nvidia', 'libEGL_nvidia']:
    for line in find_result.splitlines():
        if prefix in line and os.path.exists(line.strip()):
            nvidia_vulkan_lib = line.strip()
            break
    if nvidia_vulkan_lib:
        break

# If nothing found, install the matching NVIDIA Vulkan ICD package
if not nvidia_vulkan_lib and drv_major:
    print(f'📦 Installing NVIDIA Vulkan packages for driver {drv_major}...')
    for pkg in [f'libnvidia-gl-{drv_major}', f'nvidia-vulkan-icd',
                f'libnvidia-gl-{drv_major}-server']:
        r = subprocess.run(f'apt-get install -y -qq {pkg} 2>&1', shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print(f'  ✅ Installed {pkg}')
            break
        else:
            print(f'  ⚠️ {pkg} not available')
    # Re-search after install
    find_result2 = subprocess.run(
        'find / -name "libnvidia-vulkan-producer.so*" -o -name "libGLX_nvidia.so*" 2>/dev/null | head -10',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    for line in find_result2.splitlines():
        if os.path.exists(line.strip()):
            nvidia_vulkan_lib = line.strip()
            break

if nvidia_vulkan_lib:
    icd = {'file_format_version': '1.0.0', 'ICD': {'library_path': nvidia_vulkan_lib, 'api_version': '1.3'}}
    with open('/etc/vulkan/icd.d/nvidia_icd.json', 'w') as f:
        json.dump(icd, f)
    # Remove any existing Mesa/llvmpipe ICDs so they don't interfere
    for mesa_icd in glob.glob('/usr/share/vulkan/icd.d/*intel*') + glob.glob('/usr/share/vulkan/icd.d/*radeon*') + glob.glob('/usr/share/vulkan/icd.d/*lvp*'):
        os.rename(mesa_icd, mesa_icd + '.disabled')
    print(f'✅ NVIDIA Vulkan ICD registered: {nvidia_vulkan_lib}')
else:
    print('⚠️ Could not find NVIDIA Vulkan library.')
    print('  Real-ESRGAN will run on CPU via llvmpipe (slower but functional).')
    # Still write a best-effort ICD
    icd = {'file_format_version': '1.0.0', 'ICD': {'library_path': 'libGLX_nvidia.so.0', 'api_version': '1.3'}}
    with open('/etc/vulkan/icd.d/nvidia_icd.json', 'w') as f:
        json.dump(icd, f)

# Verify Vulkan sees the GPU
vk_check = subprocess.run('vulkaninfo --summary 2>&1 | head -30', shell=True, capture_output=True, text=True)
print(f'🔍 Vulkan devices:\n{vk_check.stdout.strip()}')

# Add MongoDB repo
!curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg 2>/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse' > /etc/apt/sources.list.d/mongodb-org-7.0.list
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq mongodb-org > /dev/null 2>&1
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongod.log --dbpath /data/db
print('✅ MongoDB running')

In [ ]:
# ═══ Step 4: Download Real-CUGAN model ═══
# Real-CUGAN weights are auto-downloaded at runtime.
# upcunet_v3.py is included in the bot repo.
print('✅ Real-CUGAN ready (weights download on first use)')


In [ ]:
# ═══ Step 5: Clone Bot & Install Dependencies ═══
import os

# Clone from GitHub
!git clone https://github.com/Alaxroy121/AnimeEncoderBot.git /kaggle/working/bot

# Install Python dependencies (bot core)
!pip install -q pyrogram tgcrypto motor pymongo python-dotenv aiofiles aiohttp google-api-python-client google-auth

# Real-CUGAN only needs torch + numpy + opencv — all pre-installed on Kaggle
# Verify torch CUDA is available
import torch
assert torch.cuda.is_available(), 'CUDA not available!'
print(f'✅ Dependencies installed (Real-CUGAN ready, {torch.cuda.device_count()} CUDA devices)')


In [ ]:
# ═══ Step 6: Start the Bot ═══
import os
os.chdir('/kaggle/working/bot')

# Verify all files are present
required = ['bot.py', 'commands.py', 'callbacks.py', 'encoder.py',
            'upscaler.py', 'database.py', 'queue_manager.py',
            'utils.py', 'config.py', 'gdrive.py']
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f'❌ Missing files: {missing}')
    print('Upload them before running!')
else:
    print('✅ All files present. Starting bot...')
    !python3 bot.py

In [ ]:
# ═══ Keep-Alive (run separately if needed) ═══
import time
from IPython.display import clear_output

i = 0
while True:
    i += 1
    time.sleep(300)
    clear_output(wait=True)
    print(f'🔄 Keep-alive #{i} — session active')